# Configuration


In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(".env", override=True)  # Change to ".env.simulation" for simulation data

# Load configuration from .env
simulation_root = os.getenv("SIMULATION_DIR")
converted_dir = os.getenv("CONVERT_TARGET_DIR")
batch_checkpoint_dir = os.getenv("BATCH_CHECKPOINT_DIR")
summarized_dir = os.getenv("SUMMARIZED_TARGET_DIR")
arena_heatmaps_output = os.getenv("ARENA_HEATMAP_TARGET_DIR")

# Display loaded configuration
print("Configuration loaded from .env:")
print(f"  SIMULATION_DIR: {simulation_root}")
print(f"  CONVERT_TARGET_DIR: {converted_dir}")
print(f"  BATCH_CHECKPOINT_DIR: {batch_checkpoint_dir}")
print(f"  SUMMARIZED_TARGET_DIR: {summarized_dir}")
print(f"  ARENA_HEATMAP_TARGET_DIR: {arena_heatmaps_output}")

# Create directories if they don't exist
os.makedirs(converted_dir, exist_ok=True)
os.makedirs(batch_checkpoint_dir, exist_ok=True)
os.makedirs(summarized_dir, exist_ok=True)
os.makedirs(arena_heatmaps_output, exist_ok=True)

Configuration loaded from .env:
  SIMULATION_DIR: /Users/defdef/Library/Application Support/DefaultCompany/Sumobot/Logs/Batch/20260811_205027_batch
  CONVERT_TARGET_DIR: .analytic-cache/converted/pacing_test_sim
  BATCH_CHECKPOINT_DIR: .analytic-cache/batched/pacing_test_sim
  SUMMARIZED_TARGET_DIR: .analytic-cache/result/pacing_test_sim
  ARENA_HEATMAP_TARGET_DIR: .analytic-cache/result/pacing_test_sim/arena_heatmaps


# Data Compiling

## Convert Simulation Log to Parquet / CSV

In [2]:
from compile.log_to_parquet import ( 
    convert_all_configs
)

convert_all_configs(simulation_root, converted_dir)

DEBUG: config_folder = /Users/defdef/Library/Application Support/DefaultCompany/Sumobot/Logs/Batch/20260811_205027_batch/Bot_LLM_vs_Bot_BT/Timer_30__ActInterval_0.1__Round_BestOf3__SkillLeft_Boost__SkillRight_Boost__Pacing_lin_down_06_04_constraint_avg_bot
DEBUG: parent_name = Bot_LLM_vs_Bot_BT
DEBUG: config_name = Timer_30__ActInterval_0.1__Round_BestOf3__SkillLeft_Boost__SkillRight_Boost__Pacing_lin_down_06_04_constraint_avg_bot
[1/1] Processing Timer_30__ActInterval_0.1__Round_BestOf3__SkillLeft_Boost__SkillRight_Boost__Pacing_lin_down_06_04_constraint_avg_bot


Processing /Users/defdef/Library/Application Support/DefaultCompany/Sumobot/Logs/Batch/20260811_205027_batch/Bot_LLM_vs_Bot_BT/Timer_30__ActInterval_0.1__Round_BestOf3__SkillLeft_Boost__SkillRight_Boost__Pacing_lin_down_06_04_constraint_avg_bot: 0it [00:00, ?it/s]

✅ Saved Parquet: .analytic-cache/converted/pacing_test_sim/Bot_LLM_vs_Bot_BT/Timer_30__ActInterval_0.1__Round_BestOf3__SkillLeft_Boost__SkillRight_Boost__Pacing_lin_down_06_04_constraint_avg_bot/Timer_30__ActInterval_0.1__Round_BestOf3__SkillLeft_Boost__SkillRight_Boost__Pacing_lin_down_06_04_constraint_avg_bot.parquet


## Generate Summarization CSV

### Generate Batched CSV

Process CSVs in batches and save checkpoints

Structure: base_dir/BotA_vs_BotB/ConfigFolder/*.csv

In [ ]:
from compile.generator import batch_process_pacing_segments

import time

timebin_size = 1
batch_size = 4 # if there's 156 matchup simulation folder, it will generate 156 / 2 = 78 summarization batch csv
input_format = "auto"  # "csv", "parquet", or "auto" (auto prefers parquet over csv)

start = time.time()

batch_process_pacing_segments(
    converted_dir, 
    batch_size=batch_size,
    applied_bots=["MCTS", "NN"],
    min_pacing=0,
    max_pacing=1,
    
    checkpoint_dir=batch_checkpoint_dir)

elapsed_seconds = time.time() - start
hours, remainder = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(remainder, 60)
processing_time = f"{int(hours):02d}:{int(minutes):02d}:{seconds:.2f}"
print(f"\nProcessing Time: {processing_time}")

### Generate Final Summarization CSV from Batches

Generate timebin summaries from batched timebin checkpoints
Loads batch files and creates final summaries

In [ ]:
from compile.generator import generate_pacing_segments_from_batches

generate_pacing_segments_from_batches(batch_checkpoint_dir, summarized_dir)

# Plotting

In [ ]:
from plotting.pacing_target_analyzer import (
    plot_all_pacing_targets,
    plot_all_pacing_targets_vs_all_rest,
    plot_all_pacing_targets_vs_each_rest,
    plot_pacing_achievement_heatmap,
    plot_all_pacing_bias_curves,
    plot_pacing_achievement_vs_winrate,
)
import pandas as pd

df_pacing_segments = pd.read_parquet(f"{summarized_dir}/summary_pacing_segments.parquet")

# Focus match = a round where the opponent was ALSO an applied bot (e.g. MCTS vs NN,
# both filter-steered) rather than a normal/static "rest" bot. Excluded by default
# from the "vs rest" charts below so the rest-bot baseline isn't blended with
# filter-vs-filter rounds; set to True to fold them back in.
include_focus_match = True

pacing_chart_dirs = {
    "applied_vs_target": f"{summarized_dir}/pacing_target_charts",
    "applied_vs_all_rest": f"{summarized_dir}/pacing_vs_all_rest_charts",
    "applied_vs_each_rest": f"{summarized_dir}/pacing_vs_each_rest_charts",
    "bias_curves": f"{summarized_dir}/pacing_bias_charts",
}

# 1. Existing: applied bot's actual pacing vs its predefined target curve, per PacingTarget
#    (now annotated with mean/std/err stats and an applied-bot baseline band+line)
#    source_dir stamps each Figure with its provenance footer here, so the PDF step
#    below doesn't need to pass it again.
figs_vs_target = plot_all_pacing_targets(
    df_pacing_segments, output_dir=pacing_chart_dirs["applied_vs_target"], source_dir=simulation_root
)

# 2. Applied bot's actual pacing vs its own target curve, averaged across ALL of the
#    rest bots it faced under this PacingTarget (no opponent line - see module docstring),
#    per applied bot x PacingTarget
figs_vs_all_rest = plot_all_pacing_targets_vs_all_rest(
    df_pacing_segments, output_dir=pacing_chart_dirs["applied_vs_all_rest"], source_dir=simulation_root,
    include_focus_match=include_focus_match,
)

# 3. Extended version of #2: the SAME applied-bot segments broken out into one line
#    per individual rest bot faced instead of pooled into a single average (still no
#    opponent's own pacing line - see module docstring), per applied bot x PacingTarget
figs_vs_each_rest = plot_all_pacing_targets_vs_each_rest(
    df_pacing_segments, output_dir=pacing_chart_dirs["applied_vs_each_rest"], source_dir=simulation_root,
    include_focus_match=include_focus_match,
)

# 4. Heatmap of achievement (1 - mean |Actual - Target|, averaged per-segment
#    error, not mean-vs-mean) across every applied bot x PacingTarget combination -
#    which bot/target combos actually hit the intended pacing curve, at a glance.
fig_achievement_heatmap = plot_pacing_achievement_heatmap(
    df_pacing_segments, include_focus_match=include_focus_match
)
if fig_achievement_heatmap is not None:
    heatmap_path = f"{summarized_dir}/pacing_achievement_heatmap.png"
    fig_achievement_heatmap.savefig(heatmap_path, dpi=150, bbox_inches="tight")
    print(f"✅ Saved: {heatmap_path}")

# 5. New: signed error (Actual - Target, not absolute) over segment index, per
#    PacingTarget - shows WHEN/WHICH DIRECTION tracking drifts (overshoot vs
#    lag), which the achievement heatmap's single MAE number collapses away.
figs_bias_curves = plot_all_pacing_bias_curves(
    df_pacing_segments, output_dir=pacing_chart_dirs["bias_curves"], source_dir=simulation_root,
    include_focus_match=include_focus_match,
)

# 6. New: does tracking the target pacing curve well actually correlate with
#    winning? Bins rounds by achievement quantile and plots win rate per bin,
#    with the achievement-vs-win correlation annotated per bot.
fig_achievement_vs_winrate = plot_pacing_achievement_vs_winrate(
    df_pacing_segments, include_focus_match=include_focus_match
)
if fig_achievement_vs_winrate is not None:
    winrate_path = f"{summarized_dir}/pacing_achievement_vs_winrate.png"
    fig_achievement_vs_winrate.savefig(winrate_path, dpi=150, bbox_inches="tight")
    print(f"✅ Saved: {winrate_path}")

In [ ]:
# 7. Wire every chart from all three "vs" chart types above plus the achievement
#    heatmap, bias curves, and achievement-vs-winrate chart into a single lossless,
#    multi-page PDF (one chart per page) instead of scattering them across cells.
#    No source_dir here - each line-chart Figure was already stamped with it above
#    (the heatmap/winrate charts weren't, since they aren't tied to a single
#    PacingTarget/bot the way the others are).
from plotting.pacing_target_analyzer import build_pacing_charts_pdf

all_pacing_figs = {**figs_vs_target, **figs_vs_all_rest, **figs_vs_each_rest, **figs_bias_curves}
if fig_achievement_heatmap is not None:
    all_pacing_figs["achievement_heatmap"] = fig_achievement_heatmap
if fig_achievement_vs_winrate is not None:
    all_pacing_figs["achievement_vs_winrate"] = fig_achievement_vs_winrate
print(f"Writing {len(all_pacing_figs)} pacing charts to PDF...")

pacing_pdf_path = build_pacing_charts_pdf(
    all_pacing_figs,
    output_path=f"{summarized_dir}/pacing_all_charts.pdf",
)